# NB07 — Adversarial Core: FGM vs FreeLB vs AWP vs FGM+AWP

BanglaBERT fine-tuned with four adversarial schemes on all four task variants; best scheme per task is selected by validation macro-F1. Runs on **RunPod (GPU)**.

**fp16 is disabled here** so the manual adversarial backward passes are numerically stable; gradient-accumulation is fixed at 1. Resumable per (method, task).

In [1]:
# Ensure dependencies (fast no-op if already installed on the pod)
import importlib, subprocess, sys
def ensure(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        print(f"installing {pip_name} ...")
        subprocess.run([sys.executable,"-m","pip","install","-q",pip_name], check=False)
for pkg,imp in [("transformers","transformers"),("datasets","datasets"),
                ("accelerate","accelerate"),("scikit-learn","sklearn"),
                ("pandas","pandas"),("numpy","numpy")]:
    ensure(pkg,imp)
print("deps ok")


deps ok


In [2]:
import os, json, time, random, shutil, warnings, unicodedata, re, gc
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report)
warnings.filterwarnings("ignore")
import transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


torch: 2.8.0+cu128 | transformers: 4.40.0
device: cuda
GPU: NVIDIA RTX A5000 25.4 GB


In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("seed set:", SEED)


seed set: 42


In [4]:
def find_repo_root():
    cands = [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents)
    for c in cands:
        if (c / "01_data").exists():
            return c.resolve()
    raise RuntimeError("Could not locate repo root (no 01_data dir found). Run from inside the repo.")

ROOT       = find_repo_root()
SPLITS     = ROOT / "01_data" / "interim" / "splits"
CKPT       = ROOT / "03_checkpoints"
TABLES     = ROOT / "04_outputs" / "tables"
PRED       = ROOT / "04_outputs" / "predictions"
LOGS       = ROOT / "04_outputs" / "run_logs"
REPORTS    = ROOT / "04_outputs" / "reports"
for p in [CKPT, TABLES, PRED, LOGS, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)
print("ROOT  :", ROOT)
print("SPLITS:", SPLITS, "| exists:", SPLITS.exists())
assert SPLITS.exists(), f"Splits folder missing: {SPLITS}. Run NB01 first."


ROOT  : /workspace/Sarcasm_detection
SPLITS: /workspace/Sarcasm_detection/01_data/interim/splits | exists: True


In [5]:
# ---- normalized key + leakage guard (defense-in-depth; NB01 already enforces this) ----
_ZW = dict.fromkeys(map(ord, ["\u200b","\u200c","\u200d","\ufeff"]), None)
def norm_key(s):
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    s = re.sub(r"\s+", " ", s).strip()
    return s.casefold()

def assert_no_internal_leakage(tr, va, te, text_col="text"):
    s_tr = {norm_key(x) for x in tr[text_col]}
    s_va = {norm_key(x) for x in va[text_col]}
    s_te = {norm_key(x) for x in te[text_col]}
    n1, n2, n3 = len(s_tr & s_va), len(s_tr & s_te), len(s_va & s_te)
    assert n1 == 0, f"LEAK train/val overlap = {n1}"
    assert n2 == 0, f"LEAK train/test overlap = {n2}"
    assert n3 == 0, f"LEAK val/test overlap = {n3}"
    return {"train_val": n1, "train_test": n2, "val_test": n3}

def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def eval_from_logits(labels, logits, num_labels):
    labels = np.asarray(labels)
    preds = np.asarray(logits).argmax(-1)
    m = {"accuracy": float(accuracy_score(labels, preds)),
         "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
         "weighted_f1": float(f1_score(labels, preds, average="weighted", zero_division=0))}
    if num_labels == 2:
        m["binary_f1"]  = float(f1_score(labels, preds, average="binary", pos_label=1, zero_division=0))
        m["precision"]  = float(precision_score(labels, preds, average="binary", pos_label=1, zero_division=0))
        m["recall"]     = float(recall_score(labels, preds, average="binary", pos_label=1, zero_division=0))
    return m, preds

def make_compute_metrics(num_labels):
    def _cm(eval_pred):
        logits, labels = eval_pred
        preds = np.asarray(logits).argmax(-1)
        return {"accuracy": accuracy_score(labels, preds),
                "macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}
    return _cm

def save_predictions(path, texts, labels, logits, num_labels, extra=None):
    logits = np.asarray(logits); probs = softmax_np(logits); preds = logits.argmax(-1)
    d = {"text": [str(t) for t in texts], "gold_label": np.asarray(labels),
         "pred_label": preds, "correct": (np.asarray(labels) == preds).astype(int)}
    for k in range(num_labels): d[f"logit_{k}"] = logits[:, k]
    for k in range(num_labels): d[f"prob_{k}"]  = probs[:, k]
    df = pd.DataFrame(d)
    if extra:
        for k, v in extra.items(): df[k] = v
    df.to_csv(path, index=False)

class TorchTextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.enc = tokenizer(list(texts), truncation=True, padding="max_length",
                             max_length=max_length, return_tensors="pt")
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

def build_args(output_dir, epochs, batch, lr, wd, warmup, fp16, metric="macro_f1"):
    common = dict(output_dir=str(output_dir), num_train_epochs=epochs,
                  per_device_train_batch_size=batch, per_device_eval_batch_size=batch*2,
                  learning_rate=lr, weight_decay=wd, warmup_ratio=warmup,
                  save_strategy="epoch", logging_strategy="epoch", save_total_limit=1,
                  load_best_model_at_end=True, metric_for_best_model=metric,
                  greater_is_better=True, report_to="none", seed=SEED, data_seed=SEED,
                  fp16=fp16, bf16=False)
    try:
        return TrainingArguments(evaluation_strategy="epoch", **common)
    except TypeError:
        return TrainingArguments(eval_strategy="epoch", **common)


In [6]:
TASKS = {
    "ben_sarc_binary":     {"label_col": "label_binary",  "num_labels": 2},
    "banglasarc_binary":   {"label_col": "label_binary",  "num_labels": 2},
    "banglasarc3_binary":  {"label_col": "label_binary",  "num_labels": 2},
    "banglasarc3_ternary": {"label_col": "label_ternary", "num_labels": 3},
}

def load_task(task):
    cfg = TASKS[task]; lc = cfg["label_col"]
    def rd(split):
        df = pd.read_csv(SPLITS / f"{task}_{split}.csv")
        assert "text" in df.columns,  f"{task}_{split}: missing 'text'"
        assert lc in df.columns,      f"{task}_{split}: missing '{lc}'"
        df = df[["text", lc]].dropna(subset=["text"]).copy()
        df["text"] = df["text"].astype(str)
        df[lc] = df[lc].astype(int)
        return df
    tr, va, te = rd("train"), rd("val"), rd("test")
    leak = assert_no_internal_leakage(tr, va, te)   # hard fail on leakage
    return tr, va, te, lc, cfg["num_labels"], leak


In [7]:
# ============================================================
# Adversarial components (FGM embedding attack, AWP weight attack)
# ============================================================
class FGM:
    def __init__(self, model, epsilon=0.5, emb_name="word_embeddings"):
        self.model=model; self.epsilon=epsilon; self.emb_name=emb_name; self.backup={}
    def attack(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad and self.emb_name in name and p.grad is not None:
                self.backup[name] = p.data.clone()
                nrm = torch.norm(p.grad)
                if nrm != 0 and not torch.isnan(nrm):
                    p.data.add_(self.epsilon * p.grad / nrm)
    def restore(self):
        for name, p in self.model.named_parameters():
            if name in self.backup: p.data = self.backup[name]
        self.backup = {}

class AWP:
    def __init__(self, model, adv_eps=1e-3):
        self.model=model; self.adv_eps=adv_eps; self.backup={}
    def attack(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None and "weight" in name and "LayerNorm" not in name:
                self.backup[name] = p.data.clone()
                ng = torch.norm(p.grad); nw = torch.norm(p.data)
                if ng != 0 and not torch.isnan(ng):
                    p.data.add_(self.adv_eps * p.grad / ng * nw)
    def restore(self):
        for name, p in self.model.named_parameters():
            if name in self.backup: p.data = self.backup[name]
        self.backup = {}

class AdvTrainer(Trainer):
    """Adversarial trainer. adv_method in {fgm, freelb, awp, fgm_awp}. Runs in fp32 for stable
    manual backward; grad-accumulation is kept at 1 so accumulated grads map 1:1 to the optimizer step."""
    def __init__(self, *a, adv_method="fgm", adv_params=None, **k):
        super().__init__(*a, **k)
        self.adv_method = adv_method
        self.adv_params = adv_params or {}
        if adv_method in ("fgm", "fgm_awp"):
            self.fgm = FGM(self.model, epsilon=self.adv_params.get("fgm_epsilon", 0.5))
        if adv_method in ("awp", "fgm_awp"):
            self.awp = AWP(self.model, adv_eps=self.adv_params.get("awp_eps", 1e-3))

    def _freelb_step(self, model, inputs):
        p = self.adv_params
        K   = int(p.get("freelb_k", 3)); lr = p.get("freelb_lr", 0.1); eps = p.get("freelb_eps", 0.1)
        input_ids = inputs["input_ids"]; attn = inputs.get("attention_mask"); labels = inputs["labels"]
        emb = model.get_input_embeddings()(input_ids)
        delta = torch.zeros_like(emb).uniform_(-eps, eps).requires_grad_(True)
        total = 0.0
        for k in range(K):
            out = model(inputs_embeds=emb + delta, attention_mask=attn, labels=labels)
            loss = out.loss / K
            self.accelerator.backward(loss)        # accumulates param grads + delta grad
            total += loss.item()
            if k == K - 1: break
            g = delta.grad
            denom = torch.norm(g.view(g.size(0), -1), dim=1).view(-1,1,1).clamp(min=1e-8)
            delta = (delta + lr * g / denom).detach()
            dn = torch.norm(delta.view(delta.size(0), -1), dim=1).view(-1,1,1)
            over = (dn > eps).float()
            delta = (delta * (eps / dn.clamp(min=1e-8)) * over + delta * (1 - over)).detach().requires_grad_(True)
            emb = model.get_input_embeddings()(input_ids).detach()
        return torch.tensor(total, device=emb.device)

    def training_step(self, model, inputs, *args, **kwargs):
        model.train()
        inputs = self._prepare_inputs(inputs)
        if self.adv_method == "freelb":
            return self._freelb_step(model, inputs)
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        if self.args.n_gpu > 1: loss = loss.mean()
        self.accelerator.backward(loss)            # clean gradient
        if self.adv_method in ("fgm", "fgm_awp"):
            self.fgm.attack()
            with self.compute_loss_context_manager():
                la = self.compute_loss(model, inputs)
            if self.args.n_gpu > 1: la = la.mean()
            self.accelerator.backward(la)
            self.fgm.restore()
        if self.adv_method in ("awp", "fgm_awp"):
            self.awp.attack()
            with self.compute_loss_context_manager():
                lw = self.compute_loss(model, inputs)
            if self.args.n_gpu > 1: lw = lw.mean()
            self.accelerator.backward(lw)
            self.awp.restore()
        return loss.detach()


In [8]:
# ---- CONFIG ----
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
BATCH_SIZE = 32
EPOCHS     = 4
LR         = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE   = 2
USE_FP16   = False   # keep False for stable adversarial backward

ADV_CONFIGS = {
    "fgm_e05":   {"method": "fgm",     "params": {"fgm_epsilon": 0.5}},
    "freelb_k3": {"method": "freelb",  "params": {"freelb_k": 3, "freelb_lr": 0.1, "freelb_eps": 0.1}},
    "awp":       {"method": "awp",     "params": {"awp_eps": 1e-3}},
    "fgm_awp":   {"method": "fgm_awp", "params": {"fgm_epsilon": 0.5, "awp_eps": 1e-3}},
}
print("configs:", list(ADV_CONFIGS), "| tasks:", list(TASKS),
      "| total runs:", len(ADV_CONFIGS)*len(TASKS))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


configs: ['fgm_e05', 'freelb_k3', 'awp', 'fgm_awp'] | tasks: ['ben_sarc_binary', 'banglasarc_binary', 'banglasarc3_binary', 'banglasarc3_ternary'] | total runs: 16


In [9]:
def run_adv(cfg_tag, cfg, task):
    tr, va, te, lc, num_labels, leak = load_task(task)
    train_ds = TorchTextDataset(tr["text"], tr[lc], tokenizer, MAX_LENGTH)
    val_ds   = TorchTextDataset(va["text"], va[lc], tokenizer, MAX_LENGTH)
    test_ds  = TorchTextDataset(te["text"], te[lc], tokenizer, MAX_LENGTH)

    out_dir = CKPT / f"07_tmp_{cfg_tag}_{task}"
    if out_dir.exists(): shutil.rmtree(out_dir, ignore_errors=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    args  = build_args(out_dir, EPOCHS, BATCH_SIZE, LR, WEIGHT_DECAY, WARMUP_RATIO, USE_FP16)
    args.gradient_accumulation_steps = 1
    trainer = AdvTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                         compute_metrics=make_compute_metrics(num_labels),
                         callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
                         adv_method=cfg["method"], adv_params=cfg["params"])
    t0 = time.time(); trainer.train(); secs = round(time.time()-t0,1)

    val_logits  = trainer.predict(val_ds).predictions
    test_logits = trainer.predict(test_ds).predictions
    val_m,  _          = eval_from_logits(va[lc].values, val_logits,  num_labels)
    test_m, test_preds = eval_from_logits(te[lc].values, test_logits, num_labels)
    print(f"  {cfg_tag} x {task} -> val_macroF1={val_m['macro_f1']:.4f} test_macroF1={test_m['macro_f1']:.4f} ({secs}s)")

    save_predictions(PRED / f"07_{cfg_tag}_{task}_val_predictions.csv",
                     va["text"].values, va[lc].values, val_logits, num_labels,
                     extra={"config": cfg_tag, "task": task, "split": "val"})
    save_predictions(PRED / f"07_{cfg_tag}_{task}_test_predictions.csv",
                     te["text"].values, te[lc].values, test_logits, num_labels,
                     extra={"config": cfg_tag, "task": task, "split": "test"})
    cm = confusion_matrix(te[lc].values, test_preds).tolist()

    row = {"config": cfg_tag, "method": cfg["method"], "task": task, "num_labels": num_labels,
           "params": json.dumps(cfg["params"]), "n_train": len(tr), "n_test": len(te),
           "time_seconds": secs, "confusion_matrix": json.dumps(cm),
           "leak_train_test": leak["train_test"]}
    for k,v in val_m.items():  row[f"val_{k}"]  = v
    for k,v in test_m.items(): row[f"test_{k}"] = v

    del model, trainer; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    shutil.rmtree(out_dir, ignore_errors=True)
    return row


In [10]:
SUMMARY = TABLES / "07_adversarial_summary.csv"
rows, done = [], set()
if SUMMARY.exists():
    prev = pd.read_csv(SUMMARY); rows = prev.to_dict("records")
    done = set(zip(prev["config"], prev["task"])); print("resuming, done:", len(done))

n = 0; total = len(ADV_CONFIGS)*len(TASKS)
for cfg_tag, cfg in ADV_CONFIGS.items():
    for task in TASKS:
        n += 1
        if (cfg_tag, task) in done:
            print(f"[{n}/{total}] skip {cfg_tag} x {task}"); continue
        print(f"[{n}/{total}] {cfg_tag} x {task}")
        try:
            rows.append(run_adv(cfg_tag, cfg, task))
            pd.DataFrame(rows).to_csv(SUMMARY, index=False)
        except Exception as e:
            print("  FAILED:", repr(e))
            if rows: pd.DataFrame(rows).to_csv(SUMMARY, index=False)
            raise
print("\nALL ADVERSARIAL RUNS DONE")


resuming, done: 12
[1/16] skip fgm_e05 x ben_sarc_binary
[2/16] skip fgm_e05 x banglasarc_binary
[3/16] skip fgm_e05 x banglasarc3_binary
[4/16] skip fgm_e05 x banglasarc3_ternary
[5/16] skip freelb_k3 x ben_sarc_binary
[6/16] skip freelb_k3 x banglasarc_binary
[7/16] skip freelb_k3 x banglasarc3_binary
[8/16] skip freelb_k3 x banglasarc3_ternary
[9/16] skip awp x ben_sarc_binary
[10/16] skip awp x banglasarc_binary
[11/16] skip awp x banglasarc3_binary
[12/16] skip awp x banglasarc3_ternary
[13/16] fgm_awp x ben_sarc_binary


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.568200,0.456130,0.795472,0.795122
2,0.413600,0.435729,0.810304,0.810243
3,0.313600,0.445413,0.809914,0.809913
4,0.236500,0.499762,0.796643,0.796314


  fgm_awp x ben_sarc_binary -> val_macroF1=0.8102 test_macroF1=0.7994 (1583.4s)
[14/16] fgm_awp x banglasarc_binary


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.671300,0.602870,0.699115,0.695926
2,0.573300,0.543981,0.724399,0.723061
3,0.496100,0.530969,0.739570,0.738396
4,0.446500,0.528117,0.752212,0.752098


  fgm_awp x banglasarc3_binary -> val_macroF1=0.7521 test_macroF1=0.7421 (501.2s)
[16/16] fgm_awp x banglasarc3_ternary


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.990300,0.866262,0.602855,0.589895
2,0.800100,0.799655,0.641478,0.632563
3,0.672600,0.804666,0.637280,0.639401
4,0.581100,0.801690,0.646516,0.645885


  fgm_awp x banglasarc3_ternary -> val_macroF1=0.6459 test_macroF1=0.6573 (745.5s)

ALL ADVERSARIAL RUNS DONE


In [11]:
res = pd.read_csv(SUMMARY)
print(res[["config","task","val_macro_f1","test_macro_f1","time_seconds"]].to_string(index=False))

pivot = res.pivot_table(index="config", columns="task", values="test_macro_f1", aggfunc="first")
pivot["mean"] = pivot.mean(axis=1)
pivot = pivot.sort_values("mean", ascending=False)
pivot.to_csv(TABLES / "07_adversarial_macro_f1_pivot.csv")
print("\nTEST MACRO-F1 by config x task:\n"); print(pivot.round(4).to_string())

# best adversarial config per task (by VAL macro-F1 -> report TEST)
best = (res.sort_values("val_macro_f1", ascending=False)
           .groupby("task").first().reset_index()
           [["task","config","method","val_macro_f1","test_macro_f1"]])
best.to_csv(TABLES / "07_best_adversarial_per_task.csv", index=False)
print("\nBEST adversarial per task (selected on val):\n"); print(best.round(4).to_string(index=False))
print("\n>>> Use the 'config'/'method' winner below as ADV in NB10 cross-dataset.")


   config                task  val_macro_f1  test_macro_f1  time_seconds
  fgm_e05     ben_sarc_binary      0.807360       0.799117        1059.4
  fgm_e05   banglasarc_binary      0.988277       0.978810         204.2
  fgm_e05  banglasarc3_binary      0.764765       0.752407         253.0
  fgm_e05 banglasarc3_ternary      0.647367       0.643383         500.8
freelb_k3     ben_sarc_binary      0.807512       0.798396        1547.7
freelb_k3   banglasarc_binary      0.990584       0.976487         296.3
freelb_k3  banglasarc3_binary      0.758544       0.759271         368.3
freelb_k3 banglasarc3_ternary      0.643008       0.643165         730.0
      awp     ben_sarc_binary      0.795185       0.790198        1093.8
      awp   banglasarc_binary      0.962631       0.969556         213.4
      awp  banglasarc3_binary      0.759659       0.747107         350.6
      awp banglasarc3_ternary      0.631617       0.631616         517.8
  fgm_awp     ben_sarc_binary      0.810243       0